## Question 2:

This question involves the implementation of a fitness tracking system that is represented by the StepTracker class. A StepTracker object is created with a parameter that defines the minimum
number of steps that must be taken for a day to be considered *active*.

The StepTracker class provides a constructor and the following methods.

• addDailySteps, which accumulates information about steps, in readings taken once per day

• activeDays, which returns the number of active days

• averageSteps, which returns the average number of steps per day, calculated by dividing the total number of steps taken by the number of days tracked

The following table contains a sample code execution sequence and the corresponding results. 

| Statements and Expressions | Value Returned (blank if no value) | Comment |
| :--- | :--- | :--- |
| `StepTracker tr = new StepTracker(10000);` | | Days with at least 10,000 steps are considered active. Assume that the parameter is positive. |
| `tr.activeDays();` | 0 | No data have been recorded yet. |
| `tr.averageSteps();` | 0.0 | When no step data have been recorded, the `averageSteps` method returns 0.0. |
| `tr.addDailySteps(9000);` | | This is too few steps for the day to be considered active. |
| `tr.addDailySteps(5000);` | | This is too few steps for the day to be considered active. |
| `tr.activeDays();` | 0 | No day had at least 10,000 steps. |
| `tr.averageSteps();` | 7000.0 | The average number of steps per day is (14000 / 2). |
| `tr.addDailySteps(13000);` | | This represents an active day. |
| `tr.activeDays();` | 1 | Of the three days for which step data were entered, one day had at least 10,000 steps. |
| `tr.averageSteps();` | 9000.0 | The average number of steps per day is (27000 / 3). |
| `tr.addDailySteps(23000);` | | This represents an active day. |
| `tr.addDailySteps(1111);` | | This is too few steps for the day to be considered active. |
| `tr.activeDays();` | 2 | Of the five days for which step data were entered, two days had at least 10,000 steps. |
| `tr.averageSteps();` | 10222.2 | The average number of steps per day is (51111 / 5). |


In [ ]:
%%js

// GAME_RUNNER: Gamified StepTracker walkthrough with movement, NPC hints, and barriers. | hide_edit: true

import { GameControl, GameEnvBackground, Player, NPC, AiNpc, Leaderboard } from '/assets/js/GameEnginev1.1/essentials/Imports.js';
import Barrier from '/assets/js/GameEnginev1.1/essentials/Barrier.js';

class StepTrackerQuestLevel {
  constructor(gameEnv) {
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;

    const bgData = {
      name: 'step_tracker_bg',
      src: path + '/images/gamebuilder/bg/alien_planet.jpg',
      pixels: { height: 772, width: 1134 }
    };

    const playerData = {
      id: 'Step Hero',
      src: path + '/images/gamify/chillguy.png',
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: width * 0.05, y: height * 0.72 },
      pixels: { height: 512, width: 384 },
      orientation: { rows: 4, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
      downLeft: { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
      right: { row: 1, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      upRight: { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
      upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    const constructorNpc = {
      id: 'Constructor Checkpoint',
      greeting: 'Constructor goal: store minSteps and initialize totalSteps, numDays, and activeDays to 0.',
      SCALE_FACTOR: 8,
      visible: false,
      INIT_POSITION: { x: width * 0.2, y: height * 0.55 },
      hitbox: { widthPercentage: 0, heightPercentage: 0 },
      reaction: function() { if (this.dialogueSystem) { this.showReactionDialogue(); } else { console.log(this.greeting); } },
      interact: function() { if (this.dialogueSystem) { this.showRandomDialogue(); } }
    };

    const addDailyNpc = {
      id: 'addDailySteps Checkpoint',
      greeting: 'Each day: totalSteps += steps, numDays++, and if steps >= minSteps then activeDays++.',
      SCALE_FACTOR: 8,
      visible: false,
      INIT_POSITION: { x: width * 0.42, y: height * 0.55 },
      hitbox: { widthPercentage: 0, heightPercentage: 0 },
      reaction: function() { if (this.dialogueSystem) { this.showReactionDialogue(); } else { console.log(this.greeting); } },
      interact: function() { if (this.dialogueSystem) { this.showRandomDialogue(); } }
    };

    const averageNpc = {
      id: 'averageSteps Checkpoint',
      greeting: 'averageSteps() returns 0.0 if no days tracked, otherwise totalSteps / numDays as a double.',
      SCALE_FACTOR: 8,
      visible: false,
      INIT_POSITION: { x: width * 0.66, y: height * 0.55 },
      hitbox: { widthPercentage: 0, heightPercentage: 0 },
      reaction: function() { if (this.dialogueSystem) { this.showReactionDialogue(); } else { console.log(this.greeting); } },
      interact: function() { if (this.dialogueSystem) { this.showRandomDialogue(); } }
    };

    const mentorSprite = path + '/images/gamify/wizard.png'; // swap to your nighthawk sprite when ready
    const mentorNpc = {
      id: 'Step Mentor AI',
      greeting: 'Welcome to StepTracker Quest! Ask me to test your constructor, addDailySteps, activeDays, or averageSteps logic.',
      src: mentorSprite,
      SCALE_FACTOR: 7,
      ANIMATION_RATE: 10,
      pixels: { height: 185, width: 163 },
      INIT_POSITION: { x: width * 0.86, y: height * 0.08 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthPercentage: 0.2, heightPercentage: 0.2 },
      expertise: 'steptracker',
      chatHistory: [],
      dialogues: [
        'What should activeDays() return before any entries?',
        'When should activeDays increment?',
        'Why must averageSteps use double division?'
      ],
      knowledgeBase: {
        steptracker: [
          {
            question: 'What does the constructor initialize?',
            answer: 'It stores minSteps and sets totalSteps, numDays, and activeDays to zero.'
          },
          {
            question: 'What makes a day active?',
            answer: 'A day is active when steps for that day are greater than or equal to minSteps.'
          },
          {
            question: 'How do you avoid integer division in averageSteps?',
            answer: 'Cast one operand to double, for example: (double) totalSteps / numDays.'
          }
        ]
      },
      reaction: function() {
        if (this.dialogueSystem) {
          this.showReactionDialogue();
        } else {
          console.log(this.greeting);
        }
      },
      interact: function() {
        AiNpc.showInteraction(this);
      }
    };

    const barrierLowSteps = {
      id: 'barrier_low_steps',
      x: width * 0.3,
      y: height * 0.4,
      width: width * 0.06,
      height: height * 0.45,
      visible: false,
      hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
      fromOverlay: true
    };

    const barrierIntegerDivision = {
      id: 'barrier_integer_division',
      x: width * 0.58,
      y: height * 0.15,
      width: width * 0.07,
      height: height * 0.5,
      visible: false,
      hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
      fromOverlay: true
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: NPC, data: constructorNpc },
      { class: NPC, data: addDailyNpc },
      { class: NPC, data: averageNpc },
      { class: NPC, data: mentorNpc },
      { class: Barrier, data: barrierLowSteps },
      { class: Barrier, data: barrierIntegerDivision }
    ];
  }
}

export const gameLevelClasses = [StepTrackerQuestLevel];
export { GameControl };

In [ ]:
// CODE_RUNNER: Write the complete StepTracker class, including the constructor and any required instance variables and methods. Your implementation must meet all specifications and conform to the example.

public class StepTracker {
    // Private instance variables
    private int minSteps;
    private int totalSteps;
    private int numDays;
    private int activeDays;
    
    // Constructor
    public StepTracker(int minSteps) {
        this.minSteps = minSteps;
        this.totalSteps = 0;
        this.numDays = 0;
        this.activeDays = 0;
    }
    
    // addDailySteps method
    public void addDailySteps(int steps) {
        totalSteps += steps;
        numDays++;
        if (steps >= minSteps) {
            activeDays++;
        }
    }
    
    // activeDays method
    public int activeDays() {
        return activeDays;
    }
    
    // averageSteps method
    public double averageSteps() {
        if (numDays == 0) {
            return 0.0; 
        }
        return (double) totalSteps / numDays;
    }
    
    public static void main(String[] args) {
        System.out.println("=== StepTracker Solution ===");
        System.out.println();
        
        StepTracker tr = new StepTracker(10000);
        System.out.println("Created StepTracker with minSteps = 10000");
        
        System.out.println(); 
        System.out.println("tr.activeDays() = " + tr.activeDays());
        System.out.println("Expected: 0");
        
        System.out.println();
        System.out.println("tr.averageSteps() = " + tr.averageSteps());
        System.out.println("Expected: 0.0");
        
        tr.addDailySteps(9000);
        System.out.println();
        System.out.println("Added 9000 steps (not active)");
        
        tr.addDailySteps(5000);
        System.out.println("Added 5000 steps (not active)");
        
        System.out.println();
        System.out.println("tr.activeDays() = " + tr.activeDays());
        System.out.println("Expected: 0");
        
        System.out.println();
        System.out.println("tr.averageSteps() = " + tr.averageSteps());
        System.out.println("Expected: 7000.0");
        
        tr.addDailySteps(13000);
        System.out.println();
        System.out.println("Added 13000 steps (ACTIVE!)");
        
        System.out.println();
        System.out.println("tr.activeDays() = " + tr.activeDays());
        System.out.println("Expected: 1");
        
        System.out.println();
        System.out.println("tr.averageSteps() = " + tr.averageSteps());
        System.out.println("Expected: 9000.0");
        
        tr.addDailySteps(23000);
        System.out.println();
        System.out.println("Added 23000 steps (ACTIVE!)");
        
        tr.addDailySteps(1111);
        System.out.println("Added 1111 steps (not active)");
        
        System.out.println();
        System.out.println("tr.activeDays() = " + tr.activeDays());
        System.out.println("Expected: 2");
        
        System.out.println();
        System.out.println("tr.averageSteps() = " + tr.averageSteps());
        System.out.println("Expected: 10222.2");
    }
}

StepTracker.main(null);

=== StepTracker Solution ===

Created StepTracker with minSteps = 10000

tr.activeDays() = 0
Expected: 0

tr.averageSteps() = 0.0
Expected: 0.0

Added 9000 steps (not active)
Added 5000 steps (not active)

tr.activeDays() = 0
Expected: 0

tr.averageSteps() = 7000.0
Expected: 7000.0

Added 13000 steps (ACTIVE!)

tr.activeDays() = 1
Expected: 1

tr.averageSteps() = 9000.0
Expected: 9000.0

Added 23000 steps (ACTIVE!)
Added 1111 steps (not active)

tr.activeDays() = 2
Expected: 2

tr.averageSteps() = 10222.2
Expected: 10222.2


## Scoring Guidelines:

**Intent:** Define implementation of a class to record fitness data

**Total:** 9 Points

**+1:** Declares all appropriate private instance variables

**+1:** Declares header: public StepTracker(int ___)

**+1:** Uses parameter and appropriate values to initialize instance variables

**+1:** Declares header: public void addDailySteps(int ___)

**+1:** Identifies active days and increments count

**+1:** Updates other instance variables appropriately

**+1:** Declares and implements public int activeDays()

**+1:** Declares header: public double averageSteps()

**+1:** Returns calculated double average number of steps

### Question 2: Scoring Notes

| Points | Rubric Criteria | Responses earn the point even if they... | Responses will not earn the point if they... |
| :--- | :--- | :--- | :--- |
| **+1** | Declares all appropriate **private** instance variables | | * omit keyword `private`<br>* declare variables outside the class |
| **+2** | **Constructor** | | |
| +1 | Declares header: `public StepTracker(int __)` | * omit keyword `public` | * declare method `private` |
| +1 | Uses parameter and appropriate values to initialize instance variables | * initialize primitive instance variables to default values when declared | * fail to use the parameter to initialize some instance variable<br>* fail to declare instance variables<br>* initialize local variables instead of instance variables<br>* assign variables to parameters |
| **+3** | **addDailySteps method** | | |
| +1 | Declares header: `public void addDailySteps(int __)` | * omit keyword `public` | * declare method `private` |
| +1 | Identifies active days and increments count | * put valid comparison erroneously in some other method | * fail to use the parameter as part of the comparison<br>* fail to increment a count of active days<br>* fail to increment an instance variable<br>* compare parameter to some numeric constant |
| +1 | Updates other instance variables appropriately | | * update another instance variable only on active days<br>* update another instance variable inappropriately<br>* fail to update appropriate instance variable<br>* update a local variable |
| **+1** | **activeDays method** | | |
| +1 | Declares and implements `public int activeDays()` | * return appropriate count of active days where the instance variables were updated improperly in `addDailySteps` or `activeDays` | * declare method `private`<br>* return value that is not the number of active days<br>* fail to return a value |
| **+2** | **averageSteps method** | | |
| +1 | Declares header: `public double averageSteps()` | * omit keyword `public` | * declare method `private` |
| +1 | Returns calculated **double** average number of steps | * maintain instance variables improperly but calculate appropriate average<br>* fail to handle the special case where no days are tracked | * use integer division<br>* calculate something other than steps divided by days<br>* fail to return |